In [19]:
from IPython.display import display, HTML
display(HTML("<style>.container {width:100% !important; }</style>"))

import glob
import pandas as pd
import numpy as np
import hvplot.pandas
import plotly.graph_objects as go

pd.options.mode.chained_assignment = None

CPU_FREQ = 2.60

rdtsc_df_dict = []
ttt_df_dict = []

# =========================
# 🔍 PARSE LOG FILES
# =========================
for filename in glob.glob("../exchange*.log") + glob.glob("../*_1.log"):
    print(f'processing {filename}')
    
    with open(filename) as f:
        for line in f:
            tokens = line.strip().split()

            if len(tokens) < 4:
                continue

            try:
                time = tokens[0]

                # Clean token (handles [RDTSC], RDTSC:, etc.)
                type_token = tokens[1].replace('[','').replace(']','').replace(':','')

                tag = tokens[2]

                # Robust latency parsing
                latency = float(tokens[-1].replace('latency=', ''))

                # Validate timestamp format
                pd.to_datetime(time, format='%H:%M:%S.%f')

            except:
                continue

            if type_token == 'RDTSC':
                rdtsc_df_dict.append({
                    'timestamp': time,
                    'tag': tag,
                    'latency': latency / CPU_FREQ
                })

            elif type_token == 'TTT':
                ttt_df_dict.append({
                    'timestamp': time,
                    'tag': tag,
                    'latency': latency
                })


# =========================
# 🧱 BUILD DATAFRAMES (SAFE)
# =========================
rdtsc_df = pd.DataFrame(rdtsc_df_dict, columns=['timestamp','tag','latency'])
ttt_df   = pd.DataFrame(ttt_df_dict, columns=['timestamp','tag','latency'])

print("RDTSC rows:", len(rdtsc_df))
print("TTT rows:", len(ttt_df))


# =========================
# 🧹 CLEAN DATA
# =========================
if not rdtsc_df.empty:
    rdtsc_df = rdtsc_df.drop_duplicates().sort_values(by='timestamp')
    rdtsc_df['timestamp'] = pd.to_datetime(rdtsc_df['timestamp'], format='%H:%M:%S.%f')
else:
    print("⚠️ No RDTSC data found!")

if not ttt_df.empty:
    ttt_df = ttt_df.drop_duplicates().sort_values(by='timestamp')
    ttt_df['timestamp'] = pd.to_datetime(ttt_df['timestamp'], format='%H:%M:%S.%f')
else:
    print("⚠️ No TTT data found!")


# =========================
# 📈 RDTSC LATENCY PLOTS
# =========================
if not rdtsc_df.empty:

    for tag in rdtsc_df['tag'].unique():
        print(tag)

        fig = go.Figure()

        t_df = rdtsc_df[rdtsc_df['tag'] == tag].copy()
        t_df = t_df[t_df['latency'] > 0]

        if len(t_df) < 5:
            continue

        # Remove outliers
        q_hi = t_df['latency'].quantile(0.99)
        q_lo = t_df['latency'].quantile(0.01)
        t_df = t_df[(t_df['latency'] < q_hi) & (t_df['latency'] > q_lo)]

        mean = t_df['latency'].mean()
        print(f'{tag} has {len(t_df)} observations mean {mean}')

        rolling_window = max(1, int(len(t_df) / 100))

        use_micros = False
        if mean >= 1000:
            use_micros = True
            t_df['latency'] = t_df['latency'] / 1000

        fig.add_trace(go.Scatter(x=t_df['timestamp'], y=t_df['latency'], name=tag))
        fig.add_trace(go.Scatter(
            x=t_df['timestamp'],
            y=t_df['latency'].rolling(rolling_window).mean(),
            name=tag + ' mean'
        ))

        fig.update_layout(
            title=f'performance {tag} {"microseconds" if use_micros else "nanoseconds"}',
            height=700,
            width=1000,
            hovermode='x'
        )

        fig.write_html(f"{tag}.html")


# =========================
# 🔗 HOP LATENCY ANALYSIS
# =========================
HOPS = [
    ['T0_OrderServer_TCP_read', 'T2_OrderServer_LFQueue_write'],
    ['T1_OrderServer_LFQueue_write', 'T3_MatchingEngine_LFQueue_read'],
    ['T2_MatchingEngine_LFQueue_read', 'T4_MatchingEngine_LFQueue_write'],
    ['T3_MatchingEngine_LFQueue_read', 'T4t_MatchingEngine_LFQueue_write'],
    ['T3_MatchingEngine_LFQueue_write', 'T5_MarketDataPublisher_LFQueue_read'],
    ['T4t_MatchingEngine_LFQueue_write', 'T5t_OrderServer_LFQueue_read'],
    ['T4_MarketDataPublisher_LFQueue_read', 'T6_MarketDataPublisher_UDP_write'],
    ['T5t_OrderServer_LFQueue_read', 'T6t_OrderServer_TCP_write'],
    ['T6_MarketDataConsumer_UDP_read', 'T8_MarketDataConsumer_LFQueue_write'],
    ['T7t_OrderGateway_TCP_read', 'T8t_OrderGateway_LFQueue_write'],
    ['T7_MarketDataConsumer_LFQueue_write', 'T9_TradeEngine_LFQueue_read'],
    ['T8t_OrderGateway_LFQueue_write', 'T9t_TradeEngine_LFQueue_read'],
    ['T8_TradeEngine_LFQueue_read', 'T10_TradeEngine_LFQueue_write'],
    ['T9t_TradeEngine_LFQueue_read', 'T10_TradeEngine_LFQueue_write'],
    ['T9_TradeEngine_LFQueue_write', 'T11_OrderGateway_LFQueue_read'],
    ['T10_OrderGateway_LFQueue_read', 'T12_OrderGateway_TCP_write'],
    ['T11_OrderGateway_TCP_write', 'T1_OrderServer_TCP_read'],
    ['T5_MarketDataPublisher_UDP_write', 'T7_MarketDataConsumer_UDP_read'],
    ['T6t_OrderServer_TCP_write', 'T7t_OrderGateway_TCP_read'],
]

if not ttt_df.empty:

    for tag_p, tag_n in HOPS:
        print(f'{tag_p} => {tag_n}')

        fig = go.Figure()

        t_df = ttt_df[(ttt_df['tag'] == tag_n) | (ttt_df['tag'] == tag_p)].copy()

        if len(t_df) < 5:
            continue

        t_df['latency_diff'] = t_df['latency'].diff()
        t_df = t_df[t_df['latency_diff'] > 0]
        t_df = t_df[t_df['tag'] == tag_n]

        if len(t_df) < 5:
            continue

        # Remove outliers
        q_hi = t_df['latency_diff'].quantile(0.99)
        q_lo = t_df['latency_diff'].quantile(0.01)
        t_df = t_df[(t_df['latency_diff'] < q_hi) & (t_df['latency_diff'] > q_lo)]

        mean = t_df['latency_diff'].mean()
        print(f'{tag_n} mean latency: {mean}')

        rolling_window = max(1, int(len(t_df) / 100))

        unit = 'nanoseconds'
        if mean >= 1_000_000:
            unit = 'milliseconds'
            t_df['latency_diff'] /= 1_000_000
        elif mean >= 1000:
            unit = 'microseconds'
            t_df['latency_diff'] /= 1000

        tag_name = f'{tag_p} -> {tag_n}'

        fig.add_trace(go.Scatter(x=t_df['timestamp'], y=t_df['latency_diff'], name=tag_name))
        fig.add_trace(go.Scatter(
            x=t_df['timestamp'],
            y=t_df['latency_diff'].rolling(rolling_window).mean(),
            name=tag_name + ' mean'
        ))

        fig.update_layout(
            title=f'performance {tag_name} {unit}',
            height=700,
            width=1000,
            hovermode='x'
        )

    fig.write_html(f"{tag}.html")

processing ../exchange_main.log
processing ../exchange_matching_engine.log
processing ../exchange_market_data_publisher.log
processing ../exchange_snapshot_synthesizer.log
processing ../exchange_order_server.log
processing ../trading_main_1.log
processing ../trading_market_data_consumer_1.log
processing ../trading_engine_1.log
processing ../trading_order_gateway_1.log
RDTSC rows: 35921
TTT rows: 32301
Exchange_FIFOSequencer_addClientRequest
Exchange_FIFOSequencer_addClientRequest has 1802 observations mean 172.78109792538208
Exchange_FIFOSequencer_sequenceAndPublish
Exchange_FIFOSequencer_sequenceAndPublish has 2742 observations mean 24954.260786624025
Exchange_MEOrderBook_checkForMatch
Exchange_MEOrderBook_checkForMatch has 958 observations mean 69625.1445318773
Exchange_MEOrderBook_addOrder
Exchange_MEOrderBook_addOrder has 691 observations mean 835.5727485249917
Exchange_TCPSocket_send
Exchange_TCPSocket_send has 2685 observations mean 101.70892422289072
Exchange_MEOrderBook_add
Exc

In [18]:
import session_info
session_info.show()